# E2 — the same classifier, six epochs

E1 stopped before it was done. Its validation loss was still falling at epoch 2.9 (6.2467 ->
6.2280 -> 6.2220) with training loss at 6.13 against validation 6.22, so the model was
underfitted, not overfitted -- there is no gap for early stopping to protect.

This run changes exactly one thing: six epochs instead of three, under one learning-rate
schedule. Resuming E1's checkpoint was the cheaper option and was rejected: its schedule had
already decayed to 4.3e-06, so continuing means a second schedule and a discontinuity in the
middle of a curve that goes into the thesis as a figure.

Everything else is E1's: t5-small encoder, 1,000 solvent and catalyst classes plus a "not
specified" class, 200 temperature values with the absent rows masked out of that head's loss,
the 486,330-row corpus, the 5,687-row clean test, effective batch 64.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob, json

train_file = next(glob.iglob("/kaggle/input/**/conditions_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/conditions_val.jsonl", recursive=True))
test_file = next(glob.iglob("/kaggle/input/**/conditions_test_clean.jsonl", recursive=True))

rows = [json.loads(line) for line in open(train_file)]
test_rows = [json.loads(line) for line in open(test_file)]
print(f"train {len(rows)} | val {sum(1 for _ in open(val_file))} | test {len(test_rows)}")

assert "full_reactants_smiles" in rows[0], "dataset is the pre-roles one"
assert len(rows) == 486330, f"expected the 486,330-row corpus, got {len(rows)}"
assert len(test_rows) == 5687, "wrong test file"

test_products = {r["product_smiles"] for r in test_rows}
leaks = sum(1 for r in rows if r["product_smiles"] in test_products)
print("rows whose product is in the test:", leaks)
assert leaks == 0, f"{leaks} leaked rows"

del rows, test_rows, test_products

base_model = "t5-small"
learning_rate = 5e-4
condition_fields = "solvent,catalyst,temperature_celsius"
num_classes = "solvent=1000,catalyst=1000,temperature_celsius=200"
output_dir = "/kaggle/working/e2_classifier_6ep"
num_train_epochs = 6
per_device_batch = 32
grad_accum = 1
time_budget_minutes = 520

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

!torchrun --nproc_per_node=2 scripts/train_conditions_classifier.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_classifier \
    --reactants-field full_reactants_smiles \
    --condition-fields "{condition_fields}" \
    --num-classes "{num_classes}" \
    --max-source-length 256 \
    --per-device-train-batch-size {per_device_batch} \
    --per-device-eval-batch-size {per_device_batch} \
    --gradient-accumulation-steps {grad_accum} \
    --learning-rate {learning_rate} \
    --num-train-epochs {num_train_epochs} \
    --eval-steps 1000 --save-steps 1000 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done or paused at the budget; tail of log:")
!tail -8 "{log_path}"

In [ ]:
import json

meta = json.load(open(f"{output_dir}/final/classifier_meta.json"))
print("base:", meta["base_model"], "| reactant side:", meta["reactants_field"])
assert meta["reactants_field"] == "full_reactants_smiles", "trained on the substrates-only column"
for field, vocabulary in meta["vocabularies"].items():
    print(
        f"  {field}: {len(vocabulary['classes'])} classes, covering "
        f"{vocabulary['coverage_of_named_rows']:.1%} of the {vocabulary['train_rows_with_value']} "
        f"rows that name one"
    )
assert meta["vocabularies"]["catalyst"]["absent_class"] == 0
assert meta["vocabularies"]["temperature_celsius"]["absent_class"] is None

In [ ]:
import json

state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[:: max(1, len(points) // 14)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
print(f"  last three: {[(round(e, 2), round(l, 4)) for e, l in points[-3:]]}")
print(f"  E1 (t5-small, 3 epochs) ended at 6.2220 and was still falling")

In [ ]:
!python scripts/evaluate_conditions_classifier.py \
    --model-dir "{output_dir}/final" \
    --test-file "{test_file}" \
    --top-k 5 --batch-size 64 --device cuda \
    --output "/kaggle/working/E2_classifier_6ep_clean_topk.json"

In [ ]:
import json

# E1 is the run this one has to beat: t5-small encoder, three epochs, same corpus and test.
REFERENCE = {
    "solvent_exact_match_top5": "E1 61.8",
    "solvent_same_group_top5": "E1 72.9",
    "catalyst_exact_match_top5": "E1 76.7",
    "catalyst_same_group_top5": "E1 90.0",
    "catalyst_record_level_top5": "E1 93.8",
    "temperature_celsius_within_tol_top5": "E1 82.6",
    "temperature_celsius_same_bucket_top5": "E1 88.2",
}

summary = json.load(open("/kaggle/working/E2_classifier_6ep_clean_topk.json"))["summary"]
print(f'{"metric":<40}{"this run":>10}{"E1":>10}')
for key, e1 in REFERENCE.items():
    print(f"{key:<40}{summary.get(key, 0) * 100:>9.1f}%{e1:>10}")

assert summary["solvent_expected_count"] == 5077
assert summary["catalyst_expected_count"] == 941
assert summary["temperature_celsius_expected_count"] == 1579